In [1]:
import os, sys, zipfile, subprocess
from pathlib import Path
from google.colab import drive

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pytorch-msssim'])

PROJECT = Path('/content/drive/MyDrive/ImageHarmonization/iHarmony4_dataset_resize_256')
DATA = Path('/content/iharmony4_256')
DATA.mkdir(parents=True, exist_ok=True)

ARCHIVES = ['HCOCO', 'HAdobe5k', 'HFlickr', 'Hday2night']
for subset in ARCHIVES:
    dest = DATA / subset
    if not (dest / '.extracted_ok').exists():
        zip_path = PROJECT / f'{subset}_256.zip'
        assert zip_path.is_file(), f'Không tìm thấy {zip_path}'
        print(f'Đang giải nén {zip_path.name}...')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(DATA)
        (dest / '.extracted_ok').touch()

print('Dữ liệu 256x256 sẵn sàng tại:', DATA)

Mounted at /content/drive
Đang giải nén HCOCO_256.zip...
Đang giải nén HAdobe5k_256.zip...
Đang giải nén HFlickr_256.zip...
Đang giải nén Hday2night_256.zip...
Dữ liệu 256x256 sẵn sàng tại: /content/iharmony4_256


In [2]:
import os
import glob
import shutil
from google.colab import drive

# ================= SỬA ĐƯỜNG DẪN DATASET CỦA BẠN =================
# Nếu dataset nằm trên Drive, sửa thành: "/content/drive/MyDrive/iharmony4_256"
DATASET_ROOT = "/content/iharmony4_256"
# =================================================================

REPO_DIR = "/content/RainNet"

# 2. Clone repo RainNet chính chủ
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/junleen/RainNet.git {REPO_DIR}
!pip install -q gdown

# 3. Tạo file IHD_test.txt ở thư mục gốc của DATASET_ROOT (phòng trường hợp evaluate.py cần)
subsets = ["HCOCO", "HAdobe5k", "HFlickr", "Hday2night"]
ihd_test_path = os.path.join(DATASET_ROOT, "IHD_test.txt")
if not os.path.exists(ihd_test_path):
    with open(ihd_test_path, "w") as out_f:
        for sub in subsets:
            sub_txt = os.path.join(DATASET_ROOT, sub, f"{sub}_test.txt")
            if os.path.exists(sub_txt):
                with open(sub_txt, "r") as in_f:
                    for line in in_f:
                        line = line.strip()
                        if line:
                            if not line.startswith(sub + "/"):
                                line = f"{sub}/{line}"
                            out_f.write(line + "\n")
    print(f"Đã tạo {ihd_test_path}")

# 4. Tải checkpoint pre-trained (bản 256x256 chuẩn trong mục Preparation #4 của README)
ckpt_dir = os.path.join(REPO_DIR, "checkpoints", "experiment_train")
os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs("/content/rainnet_download", exist_ok=True)

# Tải cả thư mục checkpoint từ Google Drive của tác giả
!gdown --folder "https://drive.google.com/drive/folders/1NMvHbnD1kW-j1KKMxEb9R9IR5drMK3GQ?usp=sharing" -O /content/rainnet_download

# Tìm file net_G_last.pth (README lưu ý: phải dùng net_G_last.pth, không phải net_G.pth)
pth_files = glob.glob("/content/rainnet_download/**/net_G_last.pth", recursive=True)
if not pth_files:
    pth_files = glob.glob("/content/rainnet_download/**/*.pth", recursive=True)

print("Các file checkpoint tìm thấy:", pth_files)
shutil.copy(pth_files[0], os.path.join(ckpt_dir, "net_G_last.pth"))
print("Đã nạp checkpoint vào:", os.path.join(ckpt_dir, "net_G_last.pth"))

Cloning into '/content/RainNet'...
remote: Enumerating objects: 130, done.
remote: Counting objects: 100% (130/130), done.
remote: Compressing objects: 100% (96/96), done.
remote: Total 130 (delta 42), reused 120 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (130/130), 9.08 MiB | 12.79 MiB/s, done.
Resolving deltas: 100% (42/42), done.
Đã tạo /content/iharmony4_256/IHD_test.txt
Retrieving folder contents
Retrieving folder 1E0FKgw9H1M67r6Vyk_-GRLzWxxE2be07 512Resolution
Processing file 1nVJFQ1iAGMeZ-ZybJm9vBQfvYs6tkctZ netG_latest.pth
Processing file 13ufyltJNUcw3lY3rxUj9vzEDgqBJsEqc net_G_last.pth
Processing file 17cXnK20yOtdpDW-uE4y6FStKD8Jwb0oB net_G.pth
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1nVJFQ1iAGMeZ-ZybJm9vBQfvYs6tkctZ
From (redirected): https://drive.google.com/uc?id=1nVJFQ1iAGMeZ-ZybJm9vBQfvYs6tkctZ&confirm=t&uuid=3e464340-b03c-4ee7-b2

In [6]:
eval_file = "/content/RainNet/evaluate.py"
with open(eval_file, "r") as f:
    code = f.read()

code = code.replace("from skimage.measure import", "from skimage.metrics import")

with open(eval_file, "w") as f:
    f.write(code)

%cd /content/RainNet
!python evaluate.py \
  --dataset_root /content/iharmony4_256 \
  --save_dir evaluated \
  --batch_size 16 \
  --device cuda

/content/RainNet
Traceback (most recent call last):
  File "/content/RainNet/evaluate.py", line 13, in <module>
    from util.config import cfg
  File "/content/RainNet/util/config.py", line 4, in <module>
    from yacs.config import CfgNode as CN
ModuleNotFoundError: No module named 'yacs'


In [7]:
!pip install -q yacs
%cd /content/RainNet
!python evaluate.py \
  --dataset_root /content/iharmony4_256 \
  --save_dir evaluated \
  --batch_size 16 \
  --device cuda

/content/RainNet
loading test file...
loading the model from ./checkpoints/experiment_train/net_G_last.pth
  0% 0/463 [00:00<?, ?it/s]
Traceback (most recent call last):
  File "/content/RainNet/evaluate.py", line 68, in <module>
    for i, batch_data in enumerate(tqdm(testloader)):
                         ~~~~~~~~~^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/tqdm/std.py", line 1181, in __iter__
    for obj in iterable:
               ^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 725, in __next__
    data = self._next_data()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 785, in _next_data
    data = self._dataset_fetcher.fetch(index)  # may raise StopIteration
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/content/RainNe

In [12]:
import os
import pandas as pd
import numpy as np

# 1. Tiêm thêm tính fMSE, fPSNR chính xác từ mask_rgb và thêm num_workers=2 vào evaluate.py
eval_file = "/content/RainNet/evaluate.py"
with open(eval_file, "r") as f:
    code = f.read()

# Tăng tốc DataLoader với num_workers=2
code = code.replace(
    "DataLoader(testdata, batch_size=user_args.batch_size, shuffle=False, drop_last=False)",
    "DataLoader(testdata, batch_size=user_args.batch_size, shuffle=False, drop_last=False, num_workers=2)"
)

# Thêm cột fMSE, fPSNR vào CSV
code = code.replace(
    "fsave_results.writelines('image_path,foreground_ratio,MSE,PSNR\\n')",
    "fsave_results.writelines('image_path,foreground_ratio,MSE,fMSE,PSNR,fPSNR\\n')"
)

old_block = """            mse_score_op = mean_squared_error(pred_rgb, real_rgb)
            psnr_score_op = peak_signal_noise_ratio(pred_rgb, real_rgb)
            all_psnr.append(psnr_score_op)
            all_mse.append(mse_score_op)

            fsave_results.writelines('%s,%.2f,%.2f,%.2f\\n' % (img_path, mask[img_idx].mean().item(), mse_score_op, psnr_score_op))"""

new_block = """            mse_score_op = mean_squared_error(pred_rgb, real_rgb)
            psnr_score_op = peak_signal_noise_ratio(pred_rgb, real_rgb)

            fg_mask = (mask_rgb[:, :, 0:1] > 127).astype(np.float32)
            fg_pixels = max(fg_mask.sum() * 3.0, 1.0)
            fmse_score_op = np.sum(((pred_rgb.astype(np.float32) - real_rgb.astype(np.float32)) ** 2) * fg_mask) / fg_pixels
            fpsnr_score_op = 10.0 * np.log10((255.0 ** 2) / max(fmse_score_op, 1e-8))

            all_psnr.append(psnr_score_op)
            all_mse.append(mse_score_op)

            fsave_results.writelines('%s,%.4f,%.4f,%.4f,%.4f,%.4f\\n' % (img_path, mask[img_idx].mean().item(), mse_score_op, fmse_score_op, psnr_score_op, fpsnr_score_op))"""

assert old_block in code, "Không tìm thấy đoạn code cần thay thế!"
code = code.replace(old_block, new_block)

with open(eval_file, "w") as f:
    f.write(code)

# 2. Chạy evaluate.py (chỉ mất ~45 giây nhờ num_workers=2)
%cd /content/RainNet
!python evaluate.py \
  --dataset_root /content/iharmony4_256 \
  --save_dir evaluated \
  --batch_size 32 \
  --device cuda

# 3. Đọc file CSV vừa xuất và thống kê theo từng tập con + ALL
df = pd.read_csv("/content/RainNet/evaluated/test_results.csv")
df["subset"] = df["image_path"].apply(lambda p: [s for s in ["HCOCO", "HAdobe5k", "HFlickr", "Hday2night"] if f"/{s}/" in p][0])

print("\n" + "="*72)
print(f"{'Dataset':<12} | {'Count':<6} | {'MSE':<8} | {'fMSE':<8} | {'PSNR':<8} | {'fPSNR':<8}")
print("-"*72)
for sub in ["HCOCO", "HAdobe5k", "HFlickr", "Hday2night"]:
    sub_df = df[df["subset"] == sub]
    print(f"{sub:<12} | {len(sub_df):<6} | {sub_df['MSE'].mean():<8.2f} | {sub_df['fMSE'].mean():<8.2f} | {sub_df['PSNR'].mean():<8.2f} | {sub_df['fPSNR'].mean():<8.2f}")
print("-"*72)
print(f"{'ALL':<12} | {len(df):<6} | {df['MSE'].mean():<8.2f} | {df['fMSE'].mean():<8.2f} | {df['PSNR'].mean():<8.2f} | {df['fPSNR'].mean():<8.2f}")
print("="*72)

/content/RainNet
loading test file...
loading the model from ./checkpoints/experiment_train/net_G_last.pth
100% 232/232 [01:50<00:00,  2.11it/s]
PSNR: 35.5694  MSE: 47.2711

Dataset      | Count  | MSE      | fMSE     | PSNR     | fPSNR   
------------------------------------------------------------------------
HCOCO        | 4283   | 33.13    | 571.96   | 36.27    | 22.07   
HAdobe5k     | 2160   | 45.22    | 338.22   | 35.91    | 24.87   
HFlickr      | 828    | 125.32   | 817.47   | 31.01    | 20.62   
Hday2night   | 133    | 49.97    | 872.98   | 35.77    | 20.46   
------------------------------------------------------------------------
ALL          | 7404   | 47.27    | 536.63   | 35.57    | 22.69   


In [13]:
import os
import torch
import numpy as np
from tqdm import tqdm
from torch.utils.data import DataLoader
from util.config import cfg
from data.iharmony4_dataset import Iharmony4Dataset
from models.networks import define_G
from evaluate import load_network

%cd /content/RainNet

# 1. Load đúng network RainNet
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg.dataset_root = "/content/iharmony4_256"
net = load_network(cfg).to(device)
net.eval()

DATASET_ROOT = "/content/iharmony4_256"
subsets = ["HCOCO", "HAdobe5k", "HFlickr", "Hday2night"]

def eval_fair_pipeline(blend_bg=True):
    results = {}
    all_mse, all_fmse, all_psnr, all_fpsnr = [], [], [], []

    for sub in subsets:
        # Tạo danh sách test cho từng subset
        sub_txt = os.path.join(DATASET_ROOT, sub, f"{sub}_test.txt")
        ihd_txt = os.path.join(DATASET_ROOT, "IHD_test.txt")
        with open(sub_txt, "r") as in_f, open(ihd_txt, "w") as out_f:
            for line in in_f:
                if line.strip():
                    out_f.write(f"{sub}/composite_images/{os.path.basename(line.strip())}\n")

        dataset = Iharmony4Dataset(cfg, is_for_train=False)
        loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=2)

        sub_mse, sub_fmse, sub_psnr, sub_fpsnr = [], [], [], []

        with torch.no_grad():
            for batch in tqdm(loader, desc=f"{sub} (blend={blend_bg})", leave=False):
                comp = batch["comp"].to(device)   # [-1, 1]
                mask = batch["mask"].to(device)   # [-1, 1] hoặc [0, 1]
                real = batch["real"].to(device)   # [-1, 1]

                # Chạy inference RainNet
                pred = net.processImage(comp, mask)

                # Đưa tất cả về thang [0, 255] dạng float32 (không ép uint8 gây mất mát)
                comp_255 = ((comp + 1.0) * 127.5).clamp(0.0, 255.0)
                real_255 = ((real + 1.0) * 127.5).clamp(0.0, 255.0)
                pred_255 = ((pred + 1.0) * 127.5).clamp(0.0, 255.0)

                # Chuẩn hóa mask về {0, 1}
                m = (mask > 0).float() if mask.min() < 0 else (mask > 0.5).float()

                # Áp dụng đúng công thức ghép nền của bạn nếu blend_bg=True
                out_255 = pred_255 * m + comp_255 * (1.0 - m) if blend_bg else pred_255

                diff_sq = (out_255 - real_255) ** 2

                # MSE toàn ảnh (chia cho H*W*3)
                mse = diff_sq.mean(dim=[1, 2, 3])

                # fMSE vùng tiền cảnh (chia cho số pixel foreground * 3)
                fg_area = (m.sum(dim=[1, 2, 3]) * 3.0).clamp(min=1.0)
                fmse = (diff_sq * m).sum(dim=[1, 2, 3]) / fg_area

                # PSNR & fPSNR chuẩn
                psnr = 10.0 * torch.log10((255.0 ** 2) / mse.clamp(min=1e-8))
                fpsnr = 10.0 * torch.log10((255.0 ** 2) / fmse.clamp(min=1e-8))

                sub_mse.extend(mse.cpu().numpy())
                sub_fmse.extend(fmse.cpu().numpy())
                sub_psnr.extend(psnr.cpu().numpy())
                sub_fpsnr.extend(fpsnr.cpu().numpy())

        results[sub] = (np.mean(sub_mse), np.mean(sub_fmse), np.mean(sub_psnr), np.mean(sub_fpsnr), len(sub_mse))
        all_mse.extend(sub_mse)
        all_fmse.extend(sub_fmse)
        all_psnr.extend(sub_psnr)
        all_fpsnr.extend(sub_fpsnr)

    # Khôi phục lại IHD_test.txt 7,404 ảnh
    with open(os.path.join(DATASET_ROOT, "IHD_test.txt"), "w") as out_f:
        for sub in subsets:
            with open(os.path.join(DATASET_ROOT, sub, f"{sub}_test.txt"), "r") as in_f:
                for line in in_f:
                    if line.strip():
                        out_f.write(f"{sub}/composite_images/{os.path.basename(line.strip())}\n")

    print(f"\n>>> KẾT QUẢ RAINNET ĐO THEO PIPELINE CỦA BẠN (Ghép nền gốc: {blend_bg}) <<<")
    print("="*72)
    print(f"{'Dataset':<12} | {'Count':<6} | {'MSE':<8} | {'fMSE':<8} | {'PSNR':<8} | {'fPSNR':<8}")
    print("-"*72)
    for sub, (m, fm, p, fp, cnt) in results.items():
        print(f"{sub:<12} | {cnt:<6} | {m:<8.2f} | {fm:<8.2f} | {p:<8.2f} | {fp:<8.2f}")
    print("-"*72)
    print(f"{'ALL':<12} | {len(all_mse):<6} | {np.mean(all_mse):<8.2f} | {np.mean(all_fmse):<8.2f} | {np.mean(all_psnr):<8.2f} | {np.mean(all_fpsnr):<8.2f}")
    print("="*72)

# Chạy đo công bằng (ghép nền gốc y hệt model của bạn)
eval_fair_pipeline(blend_bg=True)

/content/RainNet
loading the model from ./checkpoints/experiment_train/net_G_last.pth
loading test file...


loading test file...


loading test file...


loading test file...



>>> KẾT QUẢ RAINNET ĐO THEO PIPELINE CỦA BẠN (Ghép nền gốc: True) <<<
Dataset      | Count  | MSE      | fMSE     | PSNR     | fPSNR   
------------------------------------------------------------------------
HCOCO        | 4283   | 32.79    | 566.30   | 36.32    | 22.11   
HAdobe5k     | 2160   | 44.89    | 335.52   | 35.95    | 24.91   
HFlickr      | 828    | 123.43   | 808.65   | 31.07    | 20.68   
Hday2night   | 133    | 49.44    | 864.64   | 35.79    | 20.48   
------------------------------------------------------------------------
ALL          | 7404   | 46.76    | 531.44   | 35.61    | 22.74   


In [14]:
import os
import shutil
from google.colab import drive

# 1. Mount Google Drive (nếu chưa mount)
drive.mount('/content/drive')

# 2. Tạo thư mục lưu trữ cho RainNet trên Drive
SAVE_DIR = "/content/drive/MyDrive/ImageHarmonization/runs/RainNet_Pretrained"
os.makedirs(SAVE_DIR, exist_ok=True)

# 3. Lưu checkpoint pre-trained net_G_last.pth
ckpt_src = "/content/RainNet/checkpoints/experiment_train/net_G_last.pth"
if os.path.exists(ckpt_src):
    shutil.copy(ckpt_src, os.path.join(SAVE_DIR, "RainNet_net_G_last.pth"))
    print("✓ Đã lưu checkpoint:", os.path.join(SAVE_DIR, "RainNet_net_G_last.pth"))

# 4. Lưu file CSV chứa điểm MSE, fMSE, PSNR, fPSNR của từng ảnh (7,404 dòng)
csv_src = "/content/RainNet/evaluated/test_results.csv"
if os.path.exists(csv_src):
    shutil.copy(csv_src, os.path.join(SAVE_DIR, "RainNet_test_results_7404.csv"))
    print("✓ Đã lưu file CSV 7,404 ảnh:", os.path.join(SAVE_DIR, "RainNet_test_results_7404.csv"))

# 5. Lưu luôn bảng tổng kết số liệu đo công bằng (blend_bg=True) thành file text để tiện tra cứu
summary_text = """KẾT QUẢ ĐÁNH GIÁ RAINNET (CHECKPOINT CÔNG KHAI CVPR 2021) - 256x256 (Ghép nền gốc)
========================================================================
Dataset      | Count  | MSE      | fMSE     | PSNR     | fPSNR
------------------------------------------------------------------------
HCOCO        | 4283   | 32.79    | 566.30   | 36.32    | 22.11
HAdobe5k     | 2160   | 44.89    | 335.52   | 35.95    | 24.91
HFlickr      | 828    | 123.43   | 808.65   | 31.07    | 20.68
Hday2night   | 133    | 49.44    | 864.64   | 35.79    | 20.48
------------------------------------------------------------------------
ALL          | 7404   | 46.76    | 531.44   | 35.61    | 22.74
========================================================================
"""

summary_path = os.path.join(SAVE_DIR, "RainNet_evaluation_summary.txt")
with open(summary_path, "w", encoding="utf-8") as f:
    f.write(summary_text)
print("✓ Đã lưu bảng tổng hợp:", summary_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Đã lưu checkpoint: /content/drive/MyDrive/ImageHarmonization/runs/RainNet_Pretrained/RainNet_net_G_last.pth
✓ Đã lưu file CSV 7,404 ảnh: /content/drive/MyDrive/ImageHarmonization/runs/RainNet_Pretrained/RainNet_test_results_7404.csv
✓ Đã lưu bảng tổng hợp: /content/drive/MyDrive/ImageHarmonization/runs/RainNet_Pretrained/RainNet_evaluation_summary.txt
